# 용늪 생물지형학 실습
### BIOME4–Pelletier 결합모델

- 계산기간: 21.0–0.0 ka BP 범위에서 선택
- 시간간격: 0.1 kyr
- 공간해상도: 20 m
- 유효 격자: 298셀
- 기후자료: CHELSA–TraCE21k
- 식생모델: BIOME4 v4.2b2
- 지형발달모델: Pelletier 계열 수치모델


In [ ]:
#@title 1. Google Colab 환경 준비 (최초 한 번 실행)
import os, sys, subprocess, importlib.util
PB4_THREADS=max(1,min(int(os.cpu_count() or 1),2))
os.environ["OMP_NUM_THREADS"] = str(PB4_THREADS)
os.environ["OMP_DYNAMIC"] = "FALSE"
os.environ["OMP_WAIT_POLICY"] = "PASSIVE"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
if subprocess.run(["bash","-lc","command -v gfortran"],capture_output=True).returncode != 0:
    subprocess.check_call(["apt-get","update","-qq"])
    subprocess.check_call(["apt-get","install","-y","-qq","gfortran"])
required={"numpy":"numpy","pandas":"pandas","matplotlib":"matplotlib","rasterio":"rasterio","scipy":"scipy","xarray":"xarray","geopandas":"geopandas>=1.0","shapely":"shapely>=2.0","pyogrio":"pyogrio","pyproj":"pyproj","netCDF4":"netCDF4","ipywidgets":"ipywidgets"}
missing=[spec for module,spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
import matplotlib.font_manager as fm
korean_fonts={"NanumGothic","Noto Sans CJK KR","Noto Sans CJK JP","Malgun Gothic"}
if not (korean_fonts & {font.name for font in fm.fontManager.ttflist}):
    subprocess.check_call(["apt-get","install","-y","-qq","fonts-nanum"])
    font_file="/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
    if os.path.isfile(font_file):
        fm.fontManager.addfont(font_file)
from pathlib import Path
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
ko_fonts=("NanumGothic","Noto Sans CJK KR","Malgun Gothic","AppleGothic")
installed={ft.name for ft in fm.fontManager.ttflist}
if not any(name in installed for name in ko_fonts):
    subprocess.check_call(["apt-get","update","-qq"])
    subprocess.check_call(["apt-get","install","-y","-qq","fonts-nanum"])
    for path in Path("/usr/share/fonts").rglob("NanumGothic*.ttf"):
        fm.fontManager.addfont(str(path))
    installed={ft.name for ft in fm.fontManager.ttflist}
font_name=next((name for name in ko_fonts if name in installed),None)
if font_name:
    plt.rcParams["font.family"]=font_name
plt.rcParams["axes.unicode_minus"]=False
print("Fortran, Python, 글꼴 준비 완료:",font_name or "기본 글꼴")
print(f"계산 자원: CPU {os.cpu_count()}개 | BIOME4 OpenMP {PB4_THREADS}스레드")

## 2. 입력자료


In [ ]:
#@title 2. 내장 자료 복원
import base64, io, zipfile, hashlib, sys
from pathlib import Path
ARCHIVE_BASE64 = """
...
BgAAAAA4ADgA9BYAAAtcAwAAAA==
"""
raw=base64.b64decode("".join(ARCHIVE_BASE64.split()))
checksum=hashlib.sha256(raw).hexdigest()
EXPECTED="93790ba804a9cbce01291015af2750974d85b9688f0894de8d51d46cfb5a4b7b"
print("내장 모델 SHA256:", checksum)
if checksum != EXPECTED:
    raise RuntimeError("내장 모델 해시와 GitHub 체크섬이 일치하지 않습니다: 원본을 확인하세요.")
ROOT=Path("/content/yongneup_biome4_teaching") if Path("/content").exists() else Path.cwd()/"yongneup_biome4_teaching"
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    archive.extractall(ROOT)
MODEL_DIR=ROOT/"PB4Studio_v6.6.3_CHELSA21K"
assert (MODEL_DIR/"pb4studio"/"runner.py").is_file()
assert (MODEL_DIR/"fortran_src"/"biome4_original_4_2b2.f").is_file()
assert (MODEL_DIR/"embedded_inputs"/"yongneup_exact20m"/"YONGNEUP_CHELSA_TRACE21k_ENVICLOUD_RAW_WIDE.csv").is_file()
# BIOME4 native biome ID를 스냅샷과 시계열 집계에 추가
runner_path=MODEL_DIR/"pb4studio"/"runner.py"
runner_text=runner_path.read_text(encoding="utf-8")

summary_marker='        row = summarize_snapshot(int(round(igrid.resolution_m)), float(ka), z, h, veg_code, npp, eemt, igrid.land)\n'
if '"biome4_full_id_counts"' not in runner_text:
    if runner_text.count(summary_marker)!=1:
        raise RuntimeError("runner.py의 BIOME4 집계 위치를 찾지 못했습니다.")
    summary_insert=summary_marker+'''        native_biome=np.asarray(
            veg.get("biome4_full_id_node",np.full(igrid.shape,np.nan)),
            dtype="float32"
        )
        native_land=native_biome[igrid.land]
        native_valid=native_land[np.isfinite(native_land)]
        if native_valid.size:
            native_ids,native_counts=np.unique(native_valid.astype("int16"),return_counts=True)
            row["dominant_biome4_id"]=int(native_ids[np.argmax(native_counts)])
            row["biome4_full_id_counts"]=";".join(
                f"{int(k)}:{int(v)}" for k,v in zip(native_ids,native_counts)
            )
        else:
            row["dominant_biome4_id"]=-1
            row["biome4_full_id_counts"]=""
'''
    runner_text=runner_text.replace(summary_marker,summary_insert,1)

snapshot_marker='            if "lai_node" in veg:\n'
if 'snapshot_fields["biome4_full_id"]' not in runner_text:
    if runner_text.count(snapshot_marker)<1:
        raise RuntimeError("runner.py의 스냅샷 출력 위치를 찾지 못했습니다.")
    runner_text=runner_text.replace(
        snapshot_marker,
        '            snapshot_fields["biome4_full_id"] = native_biome\n'+snapshot_marker,
        1
    )

runner_path.write_text(runner_text,encoding="utf-8")

if str(MODEL_DIR) not in sys.path: sys.path.insert(0,str(MODEL_DIR))
from pb4studio.config import RunConfig, RunMode, InputPaths, TimeConfig, ScienceConfig
from pb4studio import runner
print("모델 및 입력자료 복원 완료")


## 3. 시나리오 설정

| 항목 | 내용 |
|---|---|
| 기후 프리셋 | CHELSA 원자료, 한랭건조, 한랭습윤, 온난건조, 온난습윤 |
| 추가 기온 편차 | 월별 최저·최고기온에 더하는 값(°C) |
| 추가 강수 배율 | 월강수량에 곱하는 값 |
| 초기 토심 배율 | 초기 토심 지도에 곱하는 값 |
| 융기율 | 동적모델의 융기율(m/kyr) |
| 토양생산 계수 a | \(p_0=a\exp(0.030\,EEMT)\) |
| 사면수송 계수 c, d | \(K_d=c\,EEMT+d\,AGB\) |
| 유수침식 계수 K0 | \(K=K_0/EEMT\) |


In [ ]:
#@title 3. 시나리오 설정
import ipywidgets as widgets
from IPython.display import display, Markdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

scenario_count_widget=widgets.Dropdown(
    options=[2,3,4],
    value=2,
    description="시나리오 수",
    style={"description_width":"110px"},
    layout=widgets.Layout(width="300px")
)

start_ka_widget=widgets.BoundedFloatText(
    value=21.0,
    min=0.0,
    max=21.0,
    step=0.1,
    description="시작 (ka BP)",
    style={"description_width":"105px"},
    layout=widgets.Layout(width="225px")
)
end_ka_widget=widgets.BoundedFloatText(
    value=0.0,
    min=0.0,
    max=21.0,
    step=0.1,
    description="종료 (ka BP)",
    style={"description_width":"105px"},
    layout=widgets.Layout(width="225px")
)

def _slider(desc,value,minv,maxv,step,fmt=".3f"):
    return widgets.FloatSlider(
        description=desc,
        value=value,
        min=minv,
        max=maxv,
        step=step,
        readout=True,
        readout_format=fmt,
        continuous_update=False,
        style={"description_width":"165px"},
        layout=widgets.Layout(width="520px")
    )

scenario_controls=[]

for i in range(4):
    controls={
        "name":widgets.Text(
            value="baseline" if i==0 else f"scenario{i+1}",
            description="시나리오 이름",
            style={"description_width":"165px"},
            layout=widgets.Layout(width="520px")
        ),
        "model":widgets.Dropdown(
            options=["정적","동적"],
            value="동적",
            description="모델",
            style={"description_width":"165px"},
            layout=widgets.Layout(width="520px")
        ),
        "climate_type":widgets.Dropdown(
            options=["CHELSA 원자료","한랭건조","한랭습윤","온난건조","온난습윤"],
            value="CHELSA 원자료",
            description="기후 프리셋",
            style={"description_width":"165px"},
            layout=widgets.Layout(width="520px")
        ),
        "temperature_offset_c":_slider("추가 기온 편차 (°C)",0.0,-5.0,5.0,0.5,".1f"),
        "precipitation_multiplier":_slider("추가 강수 배율",1.0,0.5,1.5,0.05,".2f"),
        "initial_soil_depth_multiplier":_slider("초기 토심 배율",1.0,0.5,1.5,0.1,".1f"),
        "uplift_m_per_kyr":_slider("융기율 (m/kyr)",0.08,0.0,0.16,0.02,".2f"),
        "soil_production_a":_slider("토양생산 계수 a",0.037,0.02,0.06,0.001,".3f"),
        "hillslope_eemt_c":_slider("사면수송 계수 c",0.033,0.015,0.06,0.001,".3f"),
        "hillslope_biomass_d":_slider("사면수송 계수 d",0.050,0.02,0.10,0.005,".3f"),
        "fluvial_erodibility_k0":_slider("유수침식 계수 K0",0.020,0.01,0.04,0.002,".3f"),
    }

    panel=widgets.VBox(
        [widgets.HTML(value=f"<h3 style='margin:12px 0 6px'>시나리오 {i+1}</h3>")]
        + list(controls.values()),
        layout=widgets.Layout(
            width="100%",
            border="1px solid #dddddd",
            padding="12px",
            margin="8px 0"
        )
    )
    controls["_panel"]=panel
    scenario_controls.append(controls)

scenario_box=widgets.VBox(
    [sc["_panel"] for sc in scenario_controls],
    layout=widgets.Layout(width="100%")
)

def _update_scenario_count(change=None):
    n=int(scenario_count_widget.value)
    for i,sc in enumerate(scenario_controls):
        sc["_panel"].layout.display="flex" if i<n else "none"

scenario_count_widget.observe(_update_scenario_count,names="value")
_update_scenario_count()

display(widgets.HBox([scenario_count_widget,start_ka_widget,end_ka_widget]))
display(Markdown("### 시나리오별 설정"))
display(scenario_box)


In [ ]:
#@title 4. 시나리오 설정 확인 및 입력자료 준비
import re, shutil, json, numpy as np, pandas as pd, rasterio

scenario_count=int(scenario_count_widget.value)
start_ka=round(float(start_ka_widget.value),6)
end_ka=round(float(end_ka_widget.value),6)
if not (0.0<=end_ka<start_ka<=21.0):
    raise ValueError("계산기간은 21–0 ka BP 범위에서 시작시점이 종료시점보다 커야 합니다.")
if any(abs(x*10-round(x*10))>1e-6 for x in (start_ka,end_ka)):
    raise ValueError("시작시점과 종료시점은 0.1 kyr 단위로 입력합니다.")
period=f"{start_ka:g}–{end_ka:g} ka BP"

climate_presets={
    "CHELSA 원자료":(0.0,1.0),
    "한랭건조":(-4.0,0.5),
    "한랭습윤":(-4.0,1.5),
    "온난건조":(4.0,0.5),
    "온난습윤":(4.0,1.5),
}

scenario_configs=[]

for i in range(scenario_count):
    w=scenario_controls[i]
    name=w["name"].value.strip()

    if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}",name):
        raise ValueError(f"시나리오 {i+1} 이름은 영문자, 숫자, _ 또는 -만 사용합니다.")

    climate_type=w["climate_type"].value
    base_dT,base_P=climate_presets[climate_type]
    dT=float(base_dT)+float(w["temperature_offset_c"].value)
    p_factor=float(base_P)*float(w["precipitation_multiplier"].value)

    if not (-10 < dT < 10 and 0.2 < p_factor < 2.5):
        raise ValueError(f"{name}: 기후 조정값을 확인합니다.")

    scenario_configs.append({
        "name":name,
        "model":w["model"].value,
        "climate_type":climate_type,
        "dT_C":dT,
        "precip_multiplier":p_factor,
        "soil_depth_multiplier":float(w["initial_soil_depth_multiplier"].value),
        "uplift_m_per_kyr":float(w["uplift_m_per_kyr"].value),
        "soil_production_a":float(w["soil_production_a"].value),
        "hillslope_c":float(w["hillslope_eemt_c"].value),
        "hillslope_d":float(w["hillslope_biomass_d"].value),
        "fluvial_K0":float(w["fluvial_erodibility_k0"].value),
    })

names=[x["name"] for x in scenario_configs]
if len(set(names)) != len(names):
    raise ValueError("시나리오 이름은 서로 다르게 입력합니다.")

orig_inputs=MODEL_DIR/"embedded_inputs"/"yongneup_exact20m"
exp_root=ROOT/"teaching_experiments"
exp_root.mkdir(parents=True,exist_ok=True)

scenario_paths={}

for sc in scenario_configs:
    work=exp_root/sc["name"]
    inputs=work/"inputs"

    if inputs.exists():
        shutil.rmtree(inputs)

    shutil.copytree(orig_inputs,inputs)

    rawcsv=inputs/"YONGNEUP_CHELSA_TRACE21k_ENVICLOUD_RAW_WIDE.csv"
    clim=pd.read_csv(rawcsv)

    for month in range(1,13):
        mm=f"{month:02d}"

        for key in (f"tasmin_raw_{mm}",f"tasmax_raw_{mm}"):
            assert key in clim.columns,key
            clim[key]=clim[key].astype(float)+sc["dT_C"]

        key=f"pr_raw_{mm}"
        assert key in clim.columns,key
        clim[key]=clim[key].astype(float)*sc["precip_multiplier"]

    clim.to_csv(rawcsv,index=False,float_format="%.6f")
    assert len(clim)==211 and clim.notna().all().all()

    soil_path=inputs/"soil_depth_20m_exact_m.tif"

    with rasterio.open(soil_path) as src:
        soil_masked=src.read(1,masked=True)
        profile=src.profile.copy()
        fill=src.nodata if src.nodata is not None else -9999
        soil_array=np.asarray(soil_masked.filled(fill),dtype="float32")
        mask=np.ma.getmaskarray(soil_masked)
        soil_array[~mask]*=sc["soil_depth_multiplier"]

    # 원본 GeoTIFF가 tiled=False인 경우 GDAL block size 항목을 제거
    if not profile.get("tiled",False):
        profile.pop("blockxsize",None)
        profile.pop("blockysize",None)

    with rasterio.open(soil_path,"w",**profile) as dst:
        dst.write(soil_array,1)

    scenario_paths[sc["name"]]={
        "work":work,
        "inputs":inputs,
        "rawcsv":rawcsv,
        "soil_path":soil_path,
    }

scenario_table=pd.DataFrame(scenario_configs)

print("시나리오 수:",scenario_count,"| 기간:",period)
display(scenario_table)
print("시나리오 간 비교 기준:",scenario_configs[0]["name"])


## 5. 시나리오 일괄 실행


In [ ]:
#@title 5. 시나리오 실행
import os, time, warnings
import numpy as np
import pandas as pd
from pb4studio.pelletier_geomorph import PelletierStrictConfig

pb4_threads=int(globals().get("PB4_THREADS",max(1,min(int(os.cpu_count() or 1),2))))
times=np.round(np.arange(start_ka,end_ka-0.001,-0.1),6)
assert len(times)>1 and abs(float(times[-1])-end_ka)<1e-5
# 기간에 따라 최대 5개 대표시점 자동 저장 (첫 시점·마지막 시점 포함)
snapshot_indices=np.unique(np.rint(np.linspace(0,len(times)-1,min(5,len(times)))).astype(int))
snaps=tuple(float(times[i]) for i in snapshot_indices)

def _fmt_seconds(seconds):
    seconds=max(0,int(round(seconds)))
    h,rem=divmod(seconds,3600)
    m,s=divmod(rem,60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

scenario_outputs={}
scenario_runtime=[]
overall_start=time.perf_counter()

for scenario_no,sc in enumerate(scenario_configs,1):
    name=sc["name"]
    paths=scenario_paths[name]
    work=paths["work"]
    inputs=paths["inputs"]
    rawcsv=paths["rawcsv"]
    soil_path=paths["soil_path"]

    out_dir=work/"output"
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True,exist_ok=True)

    cfg=RunConfig(
        mode=RunMode.STATIC if sc["model"]=="정적" else RunMode.DYNAMIC,
        inputs=InputPaths(
            dem_path=str(inputs/"dem_20m_exact.tif"),
            climate_nc_path=str(rawcsv),
            soil_depth_path=str(soil_path),
            soil_texture_path=str(inputs/"soil_texture_20m_exact.tif"),
            basin_polygon_path=str(inputs/"basin_exact20m.geojson"),
            open_free_shp=str(inputs/"outlet_open_free_exact20m.geojson"),
            open_fixed_slope_shp=None,
            soil_depth_unit="m",
            validation_points_csv="",
        ),
        time=TimeConfig(
            start_ka=float(start_ka),
            end_ka=float(end_ka),
            interval_kyr=0.1,
            snapshot_ka=snaps,
        ),
        resolution_m_override=20.0,
        requested_core_count=pb4_threads,
        pelletier=PelletierStrictConfig(
            uplift_m_per_kyr=sc["uplift_m_per_kyr"],
            p0_base_m_per_kyr=sc["soil_production_a"],
            kd_c_eemt_m_per_kyr_per_mj=sc["hillslope_c"],
            kd_d_agb_m_per_kyr_per_kgm2=sc["hillslope_d"],
            k0_regolith=sc["fluvial_K0"],
            hillslope_critical_slope=1.50,
        ),
        science=ScienceConfig(
            biome4_variant="mckenzie2003",
            force_compile_biome4=False,
            validation_accuracy_method="per_output_time",
            basin_presence_min_fraction=0.01,
        ),
        output_dir=str(out_dir),
    )

    cfg.to_json(work/"run_config.json")
    settings={
        "scenario_number":scenario_no,
        "scenario_count":scenario_count,
        "scenario_name":name,
        "model":sc["model"],
        "climate_type":sc["climate_type"],
        "dT_C":sc["dT_C"],
        "precip_multiplier":sc["precip_multiplier"],
        "soil_depth_multiplier":sc["soil_depth_multiplier"],
        "start_ka":start_ka,
        "end_ka":end_ka,
        "u_m_per_kyr":sc["uplift_m_per_kyr"],
        "p0_a":sc["soil_production_a"],
        "kd_c":sc["hillslope_c"],
        "kd_d":sc["hillslope_d"],
        "K0":sc["fluvial_K0"],
        "model_sha256":checksum,
        "biome4_openmp_threads":pb4_threads,
    }
    (work/"teaching_settings.json").write_text(
        json.dumps(settings,ensure_ascii=False,indent=2),encoding="utf-8"
    )

    state={"case":None,"case_start":None}
    def progress(case_label,step_index,n_steps,ka,_name=name,_state=state):
        now=time.perf_counter()
        if _state["case"]!=case_label:
            _state["case"]=case_label
            _state["case_start"]=now
            print(f"\n[{_name} / {case_label}] 계산 시작 | {n_steps}시점",flush=True)
        elapsed=now-_state["case_start"]
        completed=max(int(step_index),0)
        if step_index==0 or step_index==n_steps-1 or step_index<10 or step_index%10==0:
            if completed>0:
                sec_per_step=elapsed/completed
                eta=sec_per_step*max(n_steps-completed,0)
                print(
                    f"[{_name} / {case_label}] {step_index+1}/{n_steps}, {ka:.1f} ka BP"
                    f" | 경과 {_fmt_seconds(elapsed)} | 남은 시간 약 {_fmt_seconds(eta)}",
                    flush=True
                )
            else:
                print(f"[{_name} / {case_label}] 1/{n_steps}, {ka:.1f} ka BP",flush=True)

    print(
        f"\n===== 시나리오 {scenario_no}/{scenario_count}: {name} ====="
        f"\n기후 {sc['climate_type']} | dT {sc['dT_C']:+.1f}°C | 강수 {sc['precip_multiplier']:.2f}배",
        flush=True
    )
    start=time.perf_counter()
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",message="Mean of empty slice",
                category=RuntimeWarning,module=r"pb4studio\.runner"
            )
            runner.run(cfg,progress_callback=progress)
    except KeyboardInterrupt:
        print(f"\n{name} 실행이 중단되었습니다. 계산 예외가 아니라 Jupyter 중단 신호입니다.",flush=True)
        raise

    elapsed=time.perf_counter()-start

    for label,subdir in [(sc["model"],"model_static" if sc["model"]=="정적" else "model_dynamic")]:
        summary_file=out_dir/subdir/"summary_timeseries.csv"
        assert summary_file.is_file(),f"{name}/{label}: summary_timeseries.csv 없음"
        summary=pd.read_csv(summary_file)
        assert len(summary)==len(times),f"{name}/{label}: 시점 수 불일치"
        assert summary["ka_bp"].is_unique,f"{name}/{label}: 중복 시점"
        assert np.allclose(summary["ka_bp"].to_numpy(float),times,atol=1e-6,rtol=0)
        for col in ("mean_elevation_m","mean_soil_depth_m","mean_npp"):
            assert np.isfinite(summary[col].to_numpy(float)).all(),f"{name}/{label}: {col} 비정상 값"

    scenario_outputs[name]=out_dir
    scenario_runtime.append({"scenario":name,"elapsed_s":elapsed})
    print(f"{name} 완료 | {_fmt_seconds(elapsed)}",flush=True)

reference_scenario=scenario_configs[0]["name"]
work=scenario_paths[reference_scenario]["work"]
inputs=scenario_paths[reference_scenario]["inputs"]
rawcsv=scenario_paths[reference_scenario]["rawcsv"]
soil_path=scenario_paths[reference_scenario]["soil_path"]
out_dir=scenario_outputs[reference_scenario]
experiment_name=reference_scenario
CASE_DIRS={"정적":"model_static","동적":"model_dynamic"}
SCENARIO_MODELS={sc["name"]:sc["model"] for sc in scenario_configs}
selected_ages=tuple(float(x) for x in snaps)

registry={
    "period":period,
    "start_ka":start_ka,
    "end_ka":end_ka,
    "reference_scenario":reference_scenario,
    "scenarios":scenario_configs,
}
registry_path=ROOT/"teaching_experiments"/"scenario_registry.json"
registry_path.write_text(json.dumps(registry,ensure_ascii=False,indent=2),encoding="utf-8")

print("\n전체 시나리오 실행 완료")
display(pd.DataFrame(scenario_runtime).assign(elapsed_min=lambda x:x.elapsed_s/60).round(2))
print("기준 시나리오:",reference_scenario)


In [ ]:
#@title 6. 시계열 비교
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

timeseries_rows=[]

for sc in scenario_configs:
    name=sc["name"]

    for model,subdir in [(sc["model"],CASE_DIRS[sc["model"]])]:
        df=pd.read_csv(scenario_outputs[name]/subdir/"summary_timeseries.csv")
        tmp=df.copy()
        tmp["scenario"]=name
        tmp["model"]=model
        timeseries_rows.append(tmp)

scenario_timeseries=pd.concat(timeseries_rows,ignore_index=True)

comparison_report=ROOT/"teaching_experiments"/"comparison_report"
comparison_report.mkdir(parents=True,exist_ok=True)
for old_name in (
    "within_scenario_dynamic_minus_static.csv",
    "between_scenario_minus_reference.csv",
    "grid_biome4_agreement.csv",
    "grid_difference_statistics.csv",
    "scenario_end_comparison.csv",
):
    old_file=comparison_report/old_name
    if old_file.exists():
        old_file.unlink()

scenario_timeseries.to_csv(
    comparison_report/"all_scenarios_timeseries.csv",
    index=False,
    encoding="utf-8-sig"
)

fig,axes=plt.subplots(1,3,figsize=(16,4.8),constrained_layout=True)

metrics=[
    ("mean_elevation_m","Mean elevation (m)"),
    ("mean_soil_depth_m","Mean soil depth (m)"),
    ("mean_npp","Mean NPP (gC m$^{-2}$ yr$^{-1}$)")
]

for ax,(col,ylabel) in zip(axes,metrics):

    for (scenario,model),g in scenario_timeseries.groupby(["scenario","model"]):
        g=g.sort_values("ka_bp",ascending=False)
        ls="-" if model=="동적" else "--"
        ax.plot(
            g["ka_bp"],
            g[col],
            linestyle=ls,
            lw=1.8,
            label=f"{scenario} / {model}"
        )

    ax.invert_xaxis()
    ax.set_xlabel("Age (ka BP)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=.2)

axes[0].legend(frameon=False,fontsize=8)
plt.show()

end_summary=scenario_timeseries[
    np.isclose(
        scenario_timeseries["ka_bp"].to_numpy(float),
        float(end_ka),
        atol=1e-6
    )
][[
    "scenario","model",
    "mean_elevation_m","mean_soil_depth_m","mean_npp"
]]

print("종료시점")
display(end_summary.round(3))


## 7. 시나리오별 공간분포와 BIOME4 생물군계

### BIOME4 생물군계 분류

| ID | BIOME4 원본 명칭 | 국문 명칭 | 생태적 특성 |
|---:|---|---|---|
| 1 | Tropical evergreen forest | 열대 상록수림 | 연중 고온다습하여 상록활엽수가 밀집한 숲. |
| 2 | Tropical semi-deciduous forest | 열대 반낙엽수림 | 열대의 계절적 건기에 일부 교목이 잎을 떨어뜨리는 숲. |
| 3 | Tropical deciduous forest/woodland | 열대 낙엽수림 및 소림 | 건기가 뚜렷하여 낙엽교목이 우세하고 수관이 비교적 개방된 숲. |
| 4 | Temperate deciduous forest | 온대 낙엽활엽수림 | 계절별 기온 변화에 따라 활엽교목이 겨울철에 낙엽하는 숲. |
| 5 | Temperate conifer forest | 온대 침엽수림 | 온대 기후에서 침엽교목이 우세한 숲. |
| 6 | Warm mixed forest | 난온대 혼효림 | 비교적 온난한 온대에서 상록성·낙엽성 교목 등이 함께 나타나는 숲. |
| 7 | Cool mixed forest | 냉온대 혼효림 | 냉량한 온대에서 낙엽활엽수와 침엽수가 함께 우세한 숲. |
| 8 | Cool conifer forest | 냉온대 침엽수림 | 냉량한 기후에서 침엽교목이 우세한 숲. |
| 9 | Cold mixed forest | 한랭 혼효림 | 한랭한 지역에서 침엽교목과 낙엽교목이 혼재하는 숲. |
| 10 | Evegreen taiga/montane forest | 상록 타이가 및 산지림 | 아한대 또는 고산의 긴 추위에 적응한 상록침엽수림. |
| 11 | Deciduous taiga/montane forest | 낙엽 타이가 및 산지림 | 아한대 또는 고산에서 낙엽성 교목이 우세한 숲. |
| 12 | Tropical savanna | 열대 사바나 | 열대의 계절적 건조 환경에서 초본층과 드문드문 선 교목이 공존하는 식생. |
| 13 | Tropical xerophytic shrubland | 열대 건생관목림 | 강수량이 적은 열대에서 건조에 적응한 관목이 우세한 식생. |
| 14 | Temperate xerophytic shrubland | 온대 건생관목림 | 건조한 온대에서 내건성 관목이 우세한 식생. |
| 15 | Temperate sclerophyll woodland | 온대 경엽수 소림 | 여름철 건조 등에 적응한 단단한 잎의 목본이 형성한 개방된 숲. |
| 16 | Temperate broadleaved savanna | 온대 활엽수 사바나 | 온대에서 초본층과 개방된 활엽교목층이 함께 나타나는 식생. |
| 17 | Open conifer woodland | 개방 침엽수림 | 침엽교목이 낮은 밀도로 분포하여 수관 사이가 개방된 숲. |
| 18 | Boreal parkland | 아한대 소림 및 개방식생 | 아한대 수목한계 부근에서 드문드문 선 교목과 저목·초본이 공존하는 식생. |
| 19 | Tropical grassland | 열대 초원 | 열대에서 화본과 초본이 주로 우세하고 교목 피복이 낮은 초원. |
| 20 | Temperate grassland | 온대 초원 | 온대에서 초본류가 우세하며 교목층이 거의 발달하지 않은 초원. |
| 21 | Desert | 사막 | 수분 부족으로 식생 피복과 생산성이 매우 낮은 지역. |
| 22 | Steppe tundra | 스텝 툰드라 | 한랭하고 비교적 건조한 조건에서 초본과 낮은 툰드라 식생이 혼재하는 지역. |
| 23 | Shrub tundra | 관목 툰드라 | 수목한계 너머의 한랭 지역에서 관목이 우세한 툰드라. |
| 24 | Dwarf shrub tundra | 왜성관목 툰드라 | 낮은 온도와 짧은 생육기에 적응한 왜성관목이 우세한 툰드라. |
| 25 | Prostrate shrub tundra | 포복관목 툰드라 | 지표 가까이 기는 관목이 우세한 한랭지 툰드라. |
| 26 | Cushion-forbs, lichen and moss | 방석형 초본·지의류·선태류 식생 | 매우 한랭한 환경에서 방석형 초본, 지의류 및 선태류가 드물게 분포하는 식생. |
| 27 | Barren | 무식생지 | 토양·기후 조건 등으로 식물이 거의 존재하지 않는 지표. |
| 28 | Land ice | 빙설 피복지 | 지표가 지속적인 얼음으로 덮여 육상식생이 형성되지 않는 지역. |

참고: [Kaplan et al. (2003), BIOME4 모형](https://pmip2.lsce.ipsl.fr/synth/biome4.shtml), [Kaplan et al. (2002), BIOME4 생물군계 표](https://doi.org/10.1029/2001GB001403).



In [ ]:
#@title 7. 시나리오별 토심·BIOME4 biome·고도·NPP 지도
comparison_map_age_ka = 0.0 #@param {type:"number"}
from pb4studio.snapshots import snapshot_dir_name
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import rasterio, numpy as np

map_age=min(selected_ages,key=lambda a:abs(a-float(comparison_map_age_ka)))

if abs(map_age-float(comparison_map_age_ka))>1e-6:
    print(f"저장시점 {map_age:g} ka를 표시합니다.")

BIOME4_NAMES={
    1:"Tropical evergreen forest",
    2:"Tropical semi-deciduous forest",
    3:"Tropical deciduous forest/woodland",
    4:"Temperate deciduous forest",
    5:"Temperate conifer forest",
    6:"Warm mixed forest",
    7:"Cool mixed forest",
    8:"Cool conifer forest",
    9:"Cold mixed forest",
    10:"Evegreen taiga/montane forest",
    11:"Deciduous taiga/montane forest",
    12:"Tropical savanna",
    13:"Tropical xerophytic shrubland",
    14:"Temperate xerophytic shrubland",
    15:"Temperate sclerophyll woodland",
    16:"Temperate broadleaved savanna",
    17:"Open conifer woodland",
    18:"Boreal parkland",
    19:"Tropical grassland",
    20:"Temperate grassland",
    21:"Desert",
    22:"Steppe tundra",
    23:"Shrub tundra",
    24:"Dwarf shrub tundra",
    25:"Prostrate shrub tundra",
    26:"Cushion-forbs, lichen and moss",
    27:"Barren",
    28:"Land ice",
}
BIOME4_NAMES_KO={
    1:"열대 상록수림",
    2:"열대 반낙엽수림",
    3:"열대 낙엽수림 및 소림",
    4:"온대 낙엽활엽수림",
    5:"온대 침엽수림",
    6:"난온대 혼효림",
    7:"냉온대 혼효림",
    8:"냉온대 침엽수림",
    9:"한랭 혼효림",
    10:"상록 타이가 및 산지림",
    11:"낙엽 타이가 및 산지림",
    12:"열대 사바나",
    13:"열대 건생관목림",
    14:"온대 건생관목림",
    15:"온대 경엽수 소림",
    16:"온대 활엽수 사바나",
    17:"개방 침엽수림",
    18:"아한대 소림 및 개방식생",
    19:"열대 초원",
    20:"온대 초원",
    21:"사막",
    22:"스텝 툰드라",
    23:"관목 툰드라",
    24:"왜성관목 툰드라",
    25:"포복관목 툰드라",
    26:"방석형 초본·지의류·선태류 식생",
    27:"무식생지",
    28:"빙설 피복지",
}
BIOME4_DESCRIPTIONS_KO={
    1:"연중 고온다습하여 상록활엽수가 밀집한 숲.",
    2:"열대의 계절적 건기에 일부 교목이 잎을 떨어뜨리는 숲.",
    3:"건기가 뚜렷하여 낙엽교목이 우세하고 수관이 비교적 개방된 숲.",
    4:"계절별 기온 변화에 따라 활엽교목이 겨울철에 낙엽하는 숲.",
    5:"온대 기후에서 침엽교목이 우세한 숲.",
    6:"비교적 온난한 온대에서 상록성·낙엽성 교목 등이 함께 나타나는 숲.",
    7:"냉량한 온대에서 낙엽활엽수와 침엽수가 함께 우세한 숲.",
    8:"냉량한 기후에서 침엽교목이 우세한 숲.",
    9:"한랭한 지역에서 침엽교목과 낙엽교목이 혼재하는 숲.",
    10:"아한대 또는 고산의 긴 추위에 적응한 상록침엽수림.",
    11:"아한대 또는 고산에서 낙엽성 교목이 우세한 숲.",
    12:"열대의 계절적 건조 환경에서 초본층과 드문드문 선 교목이 공존하는 식생.",
    13:"강수량이 적은 열대에서 건조에 적응한 관목이 우세한 식생.",
    14:"건조한 온대에서 내건성 관목이 우세한 식생.",
    15:"여름철 건조 등에 적응한 단단한 잎의 목본이 형성한 개방된 숲.",
    16:"온대에서 초본층과 개방된 활엽교목층이 함께 나타나는 식생.",
    17:"침엽교목이 낮은 밀도로 분포하여 수관 사이가 개방된 숲.",
    18:"아한대 수목한계 부근에서 드문드문 선 교목과 저목·초본이 공존하는 식생.",
    19:"열대에서 화본과 초본이 주로 우세하고 교목 피복이 낮은 초원.",
    20:"온대에서 초본류가 우세하며 교목층이 거의 발달하지 않은 초원.",
    21:"수분 부족으로 식생 피복과 생산성이 매우 낮은 지역.",
    22:"한랭하고 비교적 건조한 조건에서 초본과 낮은 툰드라 식생이 혼재하는 지역.",
    23:"수목한계 너머의 한랭 지역에서 관목이 우세한 툰드라.",
    24:"낮은 온도와 짧은 생육기에 적응한 왜성관목이 우세한 툰드라.",
    25:"지표 가까이 기는 관목이 우세한 한랭지 툰드라.",
    26:"매우 한랭한 환경에서 방석형 초본, 지의류 및 선태류가 드물게 분포하는 식생.",
    27:"토양·기후 조건 등으로 식물이 거의 존재하지 않는 지표.",
    28:"지표가 지속적인 얼음으로 덮여 육상식생이 형성되지 않는 지역.",
}
BIOME4_COLORS={
    1:"#00661a",2:"#4d801a",3:"#99661a",4:"#00c800",
    5:"#006666",6:"#001a80",7:"#ccffcc",8:"#009933",
    9:"#80334d",10:"#1a1ae6",11:"#33ccff",12:"#b3e61a",
    13:"#ffb399",14:"#ffd9b3",15:"#669900",16:"#cccc00",
    17:"#ff99e6",18:"#9b7fff",19:"#e6b34d",20:"#ffe57f",
    21:"#ffffe6",22:"#e6e600",23:"#66e680",24:"#808033",
    25:"#cc9999",26:"#b399ff",27:"#ccccb3",28:"#ccffff",
}
biome4_cmap=ListedColormap([BIOME4_COLORS[i] for i in range(1,29)])
biome4_norm=BoundaryNorm(np.arange(0.5,29.5,1),biome4_cmap.N)

def _read_map(scenario,model,var):
    p=scenario_outputs[scenario]/CASE_DIRS[model]/snapshot_dir_name(float(map_age))/(var+".tif")
    with rasterio.open(p) as src:
        return src.read(1,masked=True).filled(np.nan).astype(float)

def _present_biome_ids(arrays):
    ids=set()
    for a in arrays:
        vals=a[np.isfinite(a)]
        ids.update(int(v) for v in np.unique(vals) if 1<=int(v)<=28)
    return sorted(ids)

# 열: 시나리오, 행: 변수
n_scenarios=len(scenario_configs)
fig,axes=plt.subplots(4,n_scenarios,figsize=(4.2*n_scenarios,12),squeeze=False,constrained_layout=True)
map_variables=[
    ("soil_depth_m","토심 (m)"),
    ("biome4_full_id","BIOME4 생물군계"),
    ("elevation_m","고도 (m)"),
    ("npp","NPP")
]
all_arrays={}
for sc in scenario_configs:
    name=sc["name"]
    model=sc["model"]
    for var,_ in map_variables:
        all_arrays[(name,var)]=_read_map(name,model,var)

present=set()
for r,(var,label) in enumerate(map_variables):
    if var=="biome4_full_id":
        for c,sc in enumerate(scenario_configs):
            a=all_arrays[(sc["name"],var)]
            present.update(int(x) for x in np.unique(a[np.isfinite(a)]) if 1<=int(x)<=28)
            axes[r,c].imshow(np.ma.masked_invalid(a),cmap=biome4_cmap,norm=biome4_norm,interpolation="nearest")
    else:
        valid=np.concatenate([
            a[np.isfinite(a)] for sc in scenario_configs
            for a in [all_arrays[(sc["name"],var)]]
        ])
        vmin,vmax=float(valid.min()),float(valid.max())
        if vmin==vmax:
            vmax=vmin+1e-8
        for c,sc in enumerate(scenario_configs):
            a=all_arrays[(sc["name"],var)]
            im=axes[r,c].imshow(np.ma.masked_invalid(a),vmin=vmin,vmax=vmax,interpolation="nearest")
        fig.colorbar(im,ax=list(axes[r,:]),fraction=.02,pad=.01,label=label)

    for c,sc in enumerate(scenario_configs):
        axes[r,c].set_title(f'{sc["name"]} / {sc["model"]} — {label}',fontsize=9)
        axes[r,c].set_xticks([])
        axes[r,c].set_yticks([])

if present:
    handles=[Patch(facecolor=BIOME4_COLORS[i],edgecolor="0.35",
                   label=f"{i}. {BIOME4_NAMES_KO[i]}") for i in sorted(present)]
    fig.legend(handles=handles,loc="lower center",bbox_to_anchor=(0.5,-0.02),ncol=min(4,len(handles)),frameon=False,fontsize=8)
fig.suptitle(f"{map_age:g} ka BP",fontsize=13)
plt.show()


## 8. BIOME4 생물군계 면적비율


In [ ]:
#@title 8. BIOME4 생물군계 면적비율
import numpy as np, pandas as pd
from IPython.display import display
report_dir=comparison_report

if "biome4_full_id_counts" not in scenario_timeseries.columns:
    raise RuntimeError("BIOME4 생물군계 집계가 없습니다.")

def parse_biome_counts(value):
    counts={code:0 for code in BIOME4_NAMES}
    for token in str(value).split(";"):
        if ":" in token:
            code,count=token.strip().split(":",1)
            code=int(code)
            if code in counts:
                counts[code]+=int(float(count))
    return counts

records=[]
for row in scenario_timeseries.itertuples(index=False):
    counts=parse_biome_counts(row.biome4_full_id_counts)
    for code,num in counts.items():
        records.append({
            "scenario":row.scenario,"model":row.model,
            "ka_bp":float(row.ka_bp),"biome_id":code,
            "biome_name":BIOME4_NAMES[code],
            "biome_name_ko":BIOME4_NAMES_KO[code],
            "biome_description_ko":BIOME4_DESCRIPTIONS_KO[code],
            "cells":num,
            "area_percent":100.0*num/int(row.n_cells)
        })

biome_ratios=pd.DataFrame(records)
biome_ratios.to_csv(report_dir/"biome4_biome_ratio_all_times.csv",index=False,encoding="utf-8-sig")
selected=np.isclose(
    biome_ratios["ka_bp"].to_numpy()[:,None],
    np.asarray(selected_ages)[None,:],atol=1e-6
).any(axis=1)
pivot=biome_ratios.loc[selected].pivot_table(
    index=["ka_bp","scenario","model"],columns=["biome_id","biome_name_ko"],
    values="area_percent",aggfunc="first"
).sort_index(ascending=False)
display(pivot.loc[:,pivot.sum(axis=0)>0].round(2))


## 8A. BIOME4 PFT


In [ ]:
#@title 8A. 13개 PFT 진단표
PFT_NAMES={
    1:"열대 상록교목",2:"열대 건기낙엽교목",3:"온대 상록활엽교목",
    4:"온대 낙엽교목",5:"냉량 침엽교목",6:"아한대 상록교목",
    7:"아한대 낙엽교목",8:"C3/C4 온대 초본",9:"C4 열대 초본",
    10:"C3/C4 건조지 목본",11:"툰드라 관목",12:"한랭 초본",13:"지의류·광엽초본"
}

pft_rows=[]
chosen_rows=[]
for row in scenario_timeseries.itertuples(index=False):
    for pid,pname in PFT_NAMES.items():
        prefix=f"pft{pid:02d}_"
        pft_rows.append({
            "scenario":row.scenario,"model":row.model,"ka_bp":float(row.ka_bp),
            "pft_id":pid,"pft_name":pname,
            "potential_npp":getattr(row,prefix+"npp_mean"),
            "potential_lai":getattr(row,prefix+"lai_mean"),
            "constraint_pass_fraction":getattr(row,prefix+"constraint_pass_fraction")
        })
    counts=[int(getattr(row,f"reich_lai_sapwood_pft{k:02d}_count")) for k in range(14)]
    assert sum(counts)==int(row.n_cells)
    for pid,num in enumerate(counts):
        chosen_rows.append({
            "scenario":row.scenario,"model":row.model,"ka_bp":float(row.ka_bp),
            "pft_id":pid,"pft_name":PFT_NAMES.get(pid,"PFT 미선택"),
            "chosen_cells":num,"chosen_cells_percent":100*num/sum(counts)
        })

pft_long=pd.DataFrame(pft_rows)
chosen_pft=pd.DataFrame(chosen_rows)
pft_long.to_csv(report_dir/"pft_all_times.csv",index=False,encoding="utf-8-sig")
chosen_pft.to_csv(report_dir/"selected_pft_grid_ratio_all_times.csv",index=False,encoding="utf-8-sig")

print("종료시점 최적 PFT 선택 비율(%)")
end_chosen=chosen_pft[
    np.isclose(chosen_pft.ka_bp.to_numpy(float),float(end_ka),atol=1e-6) &
    (chosen_pft.pft_id>0)
]
display(
    end_chosen.pivot_table(
        index=["scenario","model"],columns=["pft_id","pft_name"],
        values="chosen_cells_percent",aggfunc="first"
    ).fillna(0).round(2)
)


## 9. 시나리오 비교


In [ ]:
#@title 9. 시나리오 설정과 종료시점 결과
settings_table=pd.DataFrame(scenario_configs)
end_table=scenario_timeseries[
    np.isclose(scenario_timeseries.ka_bp.to_numpy(float),float(end_ka),atol=1e-6)
][[
    "scenario","model","mean_elevation_m","mean_soil_depth_m","mean_npp",
    "mean_lai","mean_aet_mm_yr","mean_runoff_mm_yr","mean_eemt",
    "reich_lai_sapwood_agb_mean_kg_m2"
]].copy()
display(settings_table.round(4))
display(end_table.round(4))
settings_table.to_csv(report_dir/"scenario_settings.csv",index=False,encoding="utf-8-sig")
end_table.to_csv(report_dir/"scenario_end_results.csv",index=False,encoding="utf-8-sig")


## 10. 보고서 파일


In [ ]:
#@title 10. 전체 실험 보고서 자료 압축
download_report_zip = False #@param {type:"boolean"}
from zipfile import ZipFile,ZIP_DEFLATED
from pathlib import Path

template_path=comparison_report/"student_report_template.md"
template_text=f"""# 용늪 생물지형학 실습 보고서

## 1. 실험 목적
- 시나리오 수: {scenario_count}

## 2. 시나리오 설정
- 표: scenario_settings.csv
- 계산기간: {start_ka:g}–{end_ka:g} ka BP
- 시나리오별 모델 유형과 설정값:

## 3. 시계열 비교
- 표: all_scenarios_timeseries.csv
- 시나리오별 토심·고도·NPP 변화:

## 4. BIOME4 biome 및 PFT
- 표: biome4_biome_ratio_all_times.csv
- 표: pft_all_times.csv
- 표: selected_pft_grid_ratio_all_times.csv
- BIOME4 biome과 최적 PFT의 변화:

## 5. 종료시점 비교
- 표: scenario_end_results.csv
- 시나리오별 결과 비교:

## 6. 토의
- 기후·식생·토심·사면과정의 상호작용:
- 시나리오에서 변경한 조건과 유지한 조건:
- 모형 결과 해석의 한계:

## 재현성
- PB4Studio v6.6.3-CHELSA21K
- 모델 SHA256: {checksum}
- 원본: https://github.com/physical-geogrpahy-forever/yongneup_nb
"""
template_path.write_text(template_text,encoding="utf-8")

export_zip=ROOT/"teaching_experiments"/"yongneup_multi_scenario_student_report.zip"
with ZipFile(export_zip,"w",ZIP_DEFLATED) as archive:
    for file in sorted(comparison_report.glob("*")):
        if file.is_file() and file.suffix.lower() in (".csv",".png",".md",".json"):
            archive.write(file,Path("comparison_report")/file.name)
    if registry_path.is_file():
        archive.write(registry_path,"scenario_registry.json")
    for sc in scenario_configs:
        w=scenario_paths[sc["name"]]["work"]
        for fn in ("teaching_settings.json","run_config.json"):
            p=w/fn
            if p.is_file():
                archive.write(p,Path(sc["name"])/fn)

print("보고서 자료 ZIP:",export_zip)
if download_report_zip:
    try:
        from google.colab import files
        files.download(str(export_zip))
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(export_zip)))
else:
    print("내려받으려면 download_report_zip에 체크하고 이 셀을 다시 실행합니다.")
